# Fine-tune GPT-2 Vietnamese for Math Word Problems

**Goal:** fine-tune `NlpHUST/gpt2-vietnamese` to solve Vietnamese math word problems.

**Inputs:** `train.json`, `valid.json`, and the local GPT-2 model folder from Kaggle Input.

**Pipeline:** load data → SFT training → generate validation outputs → evaluate by relative error.

**Rules:** Internet OFF, no extra data/API/LLM, total runtime ≤ 3 hours.


In [1]:
import os, sys, json, math, time, re, random, hashlib, inspect, unicodedata, warnings
from collections import Counter, defaultdict
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional

import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
    StoppingCriteria,
    StoppingCriteriaList,
)

try:
    from peft import LoraConfig, TaskType, get_peft_model, PeftModel
except ImportError:
    LoraConfig = None
    TaskType = None
    get_peft_model = None
    PeftModel = None


print("Torch:", torch.__version__)
print("PEFT :", "available" if LoraConfig is not None else "not installed")
print("CUDA :", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


Torch: 2.10.0+cu128
PEFT : available
CUDA : True | GPU count: 2
0 Tesla T4
1 Tesla T4


In [2]:
# ============================================================
# 1. Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào: " + " | ".join(map(str, paths)))

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
    str(Path("dataset")),                 # local fallback
)
MODEL_NAME = str(first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
    str(Path("GPT2_vietnamese")),         # local fallback
))

TRAIN_FILE = Path(DATA_DIR) / "train.json"
VALID_FILE = Path(DATA_DIR) / "valid.json"
TEST_FILE  = Path(DATA_DIR) / "test.json"      # Phase 2

# Run mode: "phase1" -> infer on valid;  "phase2" -> infer on test
RUN_MODE = "phase1"

# Prompt & special tokens
PROMPT_TEMPLATE = "Dạng: {type}\nBài toán: {q}\nĐáp án là:"
SAFE_EOS_ID = 50256

# Working dirs
WORKING_DIR               = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
FINAL_OUTPUT_DIR          = WORKING_DIR / "gpt2_math_bql_model_only_12ep_safe_retrieval"
VALID_OUTPUT_PATH         = WORKING_DIR / "valid_output.json"
VALID_REPORT_PATH         = WORKING_DIR / "valid_report.json"
MODEL_VALID_OUTPUT_PATH   = WORKING_DIR / "model_valid_output.json"
MODEL_VALID_REPORT_PATH   = WORKING_DIR / "model_valid_report.json"
HYBRID_VALID_OUTPUT_PATH  = WORKING_DIR / "hybrid_valid_output.json"
HYBRID_VALID_REPORT_PATH  = WORKING_DIR / "hybrid_valid_report.json"
VALID_RETRIEVAL_DECISIONS_PATH = WORKING_DIR / "valid_retrieval_decisions.json"
TEST_OUTPUT_PATH          = WORKING_DIR / "test_predictions.json"
MODEL_TEST_OUTPUT_PATH    = WORKING_DIR / "model_test_predictions.json"
TEST_RETRIEVAL_DECISIONS_PATH = WORKING_DIR / "test_retrieval_decisions.json"
ABLATION_DIR              = WORKING_DIR / "ablations"

# Model-only checkpoint selection.
# This version is for source-unseen/test-lạ risk: no retrieval, but choose the
# best epoch by generated-answer score on official valid.json.
SELECT_BEST_CHECKPOINT_ON_OFFICIAL_VALID = True
CHECKPOINT_SELECTION_MAX_RECORDS = 1000
CHECKPOINT_SELECTION_MAX_NEW_TOKENS = 16
CHECKPOINT_SELECTION_NUM_BEAMS = 2
CHECKPOINT_TIE_BREAK = "earlier_step"  # earlier_step or later_step
KEEP_CHECKPOINT_EVAL_OUTPUTS = True

CHECKPOINT_EVAL_DIR = WORKING_DIR / "model_only_checkpoint_eval"
CHECKPOINT_SELECTION_REPORT_PATH = WORKING_DIR / "model_only_checkpoint_selection_report.json"
SELECTED_CHECKPOINT_INFO_PATH = WORKING_DIR / "model_only_selected_checkpoint_info.json"



# Safe hybrid retrieval.
# Training remains model-only. Retrieval is applied only AFTER model generation.
# The model-only output/report are always saved separately, so the 5.69 model-only
# result is not overwritten by retrieval.
USE_SAFE_HYBRID_RETRIEVAL = True
USE_HYBRID_ONLY_IF_VALID_IMPROVES = True  # phase1: if hybrid <= model-only, final output falls back to model-only
APPLY_RETRIEVAL_TO_TEST = True            # phase2: use conservative retrieval on test; set False for pure model-only submit

RETRIEVAL_DEBUG_SAMPLE = 30
RETRIEVAL_MIN_SOURCE_CANDIDATES = 1
RETRIEVAL_REQUIRE_SAME_TYPE = True

# Retrieval-first only for these stable-ish groups.
RETRIEVAL_FIRST_TYPE_KEYWORDS = ["Rephrased", "AnsAug"]

# Model-first for variable/perturbation groups.
MODEL_FIRST_TYPE_KEYWORDS = ["SV", "FOBAR"]

# Confidence thresholds for source/type majority retrieval.
RETRIEVAL_MIN_MAJORITY_FRAC = 0.50
RETRIEVAL_MIN_NEAREST_JACCARD = 0.20

# Very conservative fallback behavior.
RETRIEVAL_ALLOW_WHEN_MODEL_UNEXTRACTABLE = True
RETRIEVAL_ALLOW_WHEN_MODEL_AGREES = True

# Data
MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None
DROP_NON_EXTRACTABLE = True

# Answer-only data filtering.
# "dedup_only" is safest: remove repeated identical (query, answer) pairs only.
# "medium_clean" also drops diagram/outlier/conflicting samples to train faster.
# "easy_clean" is more aggressive and mainly for quick experiments.
DATA_FILTER_MODE = "dedup_only"  # choices: "none", "dedup_only", "medium_clean", "easy_clean"
MEDIUM_MAX_QUERY_CHARS = 512
MEDIUM_MAX_QUERY_NUMS  = 8
MEDIUM_MAX_RESP_OPS    = 25
EASY_MAX_QUERY_CHARS   = 384
EASY_MAX_QUERY_NUMS    = 5
EASY_MAX_RESP_OPS      = 20

# Target
STAGE1_TARGET_MODE        = "answer_only"

# Lengths
MAX_LENGTH_STAGE1       = 256
MAX_NEW_TOKENS          = 16

# Training
KAGGLE_FAST              = True
STAGE1_EPOCHS            = 12
PER_DEVICE_BATCH_SIZE    = 16 if KAGGLE_FAST else 4
GRAD_ACCUM               = 2 if KAGGLE_FAST else 8
STAGE1_LR                = 1e-3
WARMUP_RATIO             = 0.05
WEIGHT_DECAY             = 0.01
SEED                     = 42

# Fast training knobs. Eval still runs after training in the inference cell.
TRAIN_EVAL_DURING_TRAIN  = False if KAGGLE_FAST else True
USE_GRADIENT_CHECKPOINTING = False if KAGGLE_FAST else True
GROUP_BY_LENGTH          = True
USE_FUSED_ADAMW          = True
DATALOADER_NUM_WORKERS   = 4 if KAGGLE_FAST else 2
DATALOADER_PIN_MEMORY    = True

# LoRA
USE_LORA                 = True
LORA_R                   = 32
LORA_ALPHA               = 64
LORA_DROPOUT             = 0.05
LORA_TARGET_MODULES      = ["c_attn", "c_proj", "c_fc"]
LORA_MERGE_AND_SAVE_FULL = True  # keep existing inference path compatible

# Decoding
DECODE_MODE              = "beam"
NUM_BEAMS                = 2
NO_REPEAT_NGRAM          = 0
REPETITION_PENALTY       = 1.05
LENGTH_PENALTY           = 0.9
RUN_DECODING_ABLATIONS   = False
DECODING_ABLATIONS       = [
    ("greedy", "greedy", 1),
    ("beam2", "beam", 2),
    ("beam4", "beam", 4),
]

# self-consistency only
SC_N                      = 5
SC_TEMPERATURE            = 0.7
SC_TOP_P                  = 0.9

# Inference safety knobs
INFER_FP16                = True
USE_TYPE_AWARE_FEWSHOT    = False

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
seed_everything(SEED)

if USE_LORA and LoraConfig is None:
    raise ImportError(
        "Kaggle runtime does not have `peft`. "
        "Either enable Internet and run `pip install peft`, add a PEFT wheel/dataset, "
        "or set USE_LORA=False for full fine-tuning."
    )

print("TRAIN_FILE        :", TRAIN_FILE)
print("VALID_FILE        :", VALID_FILE)
print("MODEL_NAME        :", MODEL_NAME)
print("FINAL_OUTPUT_DIR  :", FINAL_OUTPUT_DIR)
print("RUN_MODE          :", RUN_MODE)
print("CKPT_SELECTION    :", SELECT_BEST_CHECKPOINT_ON_OFFICIAL_VALID, "| max_records:", CHECKPOINT_SELECTION_MAX_RECORDS)


TRAIN_FILE        : /kaggle/input/datasets/kimanh2002/dataset-math/train.json
VALID_FILE        : /kaggle/input/datasets/kimanh2002/dataset-math/valid.json
MODEL_NAME        : /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
FINAL_OUTPUT_DIR  : /kaggle/working/gpt2_math_bql_model_only_12ep_safe_retrieval
RUN_MODE          : phase1
CKPT_SELECTION    : True | max_records: 1000


In [3]:
# ============================================================
# 2. Tokenizer smoke-check
# ============================================================
def strip_trailing_safe_eos(token_ids):
    if hasattr(token_ids, "detach"):
        ids = token_ids.detach().cpu().tolist()
    else:
        ids = list(token_ids)
    while ids and ids[-1] == SAFE_EOS_ID:
        ids.pop()
    return ids

def decode_model_text(tok, token_ids) -> str:
    # The provided tokenizer maps id 50256 to a normal token string ("hue")
    # even though the task requires using 50256 as EOS/PAD. Strip it manually.
    return tok.decode(strip_trailing_safe_eos(token_ids), skip_special_tokens=True)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

probe = "Bài toán: 2+3=?\nLời giải: "
print("Vocab size:", tokenizer.vocab_size)
print("Token ids :", tokenizer(probe, add_special_tokens=False)["input_ids"][:20], "...")
print("SAFE_EOS raw decode:", repr(tokenizer.decode([SAFE_EOS_ID])))
print("SAFE_EOS stripped  :", repr(decode_model_text(tokenizer, [16, SAFE_EOS_ID])))


Vocab size: 50257
Token ids : [3108, 1329, 30, 416, 15, 23, 33, 35, 203, 6501, 800, 30, 225] ...
SAFE_EOS raw decode: 'hue'
SAFE_EOS stripped  : ','


In [4]:
# ============================================================
# 3. Data loading
# ============================================================
def load_records(path: str | Path) -> list:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        return json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()

train_records = load_records(TRAIN_FILE)
valid_records = load_records(VALID_FILE)
if MAX_TRAIN_SAMPLES:
    train_records = train_records[:MAX_TRAIN_SAMPLES]
if MAX_VALID_SAMPLES:
    valid_records = valid_records[:MAX_VALID_SAMPLES]

print("train:", len(train_records), "| valid:", len(valid_records))
print("first query:", train_records[0]["query_vi"][:160])
print("first type :", train_records[0].get("type"))


train: 95400 | valid: 1000
first query: Bridgette và Alex sắp kết hôn. Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó. Họ thuê một người phục vụ ăn uống để chuẩn bị bữa ăn cho từng vị kh
first type : GSM_AnsAug


In [5]:
# ============================================================
# 4. Answer extraction + data cleaning + target normalization
# ============================================================

# ---- regex pool (case-insensitive, full-width tolerant) ----
RE_ANCHORS_VI = [
    re.compile(r"đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"đ[áa]p\s*[áa]n\s*[:：]\s*", re.IGNORECASE),
]
RE_ANCHORS_EN = [
    re.compile(r"the\s*answer\s*is\s*[:：]?\s*", re.IGNORECASE),
    re.compile(r"####\s*"),
]
RE_BOXED = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")

# Numeric literal patterns
RE_NUM_VI_DEC = re.compile(r"-?\d+,\d{1,2}")          # 1,5
RE_NUM_EN_DEC = re.compile(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?")
RE_NUM_LOOSE  = re.compile(r"-?\d+(?:[.,]\d+)?")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}

def _clean_tail(s: str) -> str:
    s = s.strip()
    # take first non-empty line after anchor
    s = s.split("\n", 1)[0].strip()
    # strip trailing punctuation / currency
    s = re.sub(r"[.,;:。、,]+$", "", s)
    # drop "đô la", "USD", "%" trailing units
    s = re.sub(r"\s*(đô\s*la|usd|đồng|vnd|cm2?|m2?|km2?|\$|%)(?:\W.*|$)", "", s, flags=re.IGNORECASE)
    s = s.strip()
    return s

def extract_anchor_answer(text: str | None) -> str | None:
    if not text:
        return None
    # last hit per anchor wins (final answer is usually at the end)
    best_pos, best_tail = -1, None
    for pat in RE_ANCHORS_VI + RE_ANCHORS_EN:
        for m in pat.finditer(text):
            if m.end() > best_pos:
                best_pos = m.end()
                best_tail = text[m.end():]
    if best_tail is not None:
        return _clean_tail(best_tail)
    # boxed fallback (take LAST boxed)
    boxes = RE_BOXED.findall(text)
    if boxes:
        return _clean_tail(boxes[-1])
    return None

def parse_number(s: str | None) -> float | None:
    if s is None:
        return None
    t = s.strip()
    if not t:
        return None
    # pure VI decimal
    if RE_NUM_VI_DEC.fullmatch(t):
        try:
            v = float(t.replace(",", "."))
            return v if math.isfinite(v) else None
        except ValueError:
            return None
    if RE_NUM_EN_DEC.fullmatch(t):
        try:
            v = float(t)
            return v if math.isfinite(v) else None
        except ValueError:
            return None

    # Strip variable assignment: x = 5 -> 5
    m = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", t)
    if m:
        t = m.group(1).strip()

    # Reject tuples / intervals early
    if t.startswith("(") and t.endswith(")") and re.search(r"\d\s*,\s*\d", t):
        return None
    if t.startswith("[") and t.endswith("]"):
        return None

    # Strip wrappers
    for _ in range(3):
        new = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", t)
        if new == t:
            break
        t = new

    t = re.sub(r"\\text\{[^}]*\}", "", t)
    t = re.sub(r"\\mathrm\{[^}]*\}", "", t)
    t = t.replace("$", "")

    for token in ("\\,", "\\!", "\\;", "\\ ", "\\left", "\\right"):
        t = t.replace(token, "")
    for token in ("\\cdot", "\\times"):
        t = t.replace(token, "*")

    # LaTeX fractions / roots
    t = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", t)
    t = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", t)
    t = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    t = t.replace("\\pi", "pi")

    # Implicit multiplication
    t = re.sub(r"(\d)\s*(sqrt|pi|\()", r"\1*\2", t)
    t = re.sub(r"(\))\s*(sqrt|pi|\d)", r"\1*\2", t)
    t = re.sub(r"(pi)\s*(sqrt|pi|\d|\()", r"\1*\2", t)

    # Comma handling
    if re.fullmatch(r"-?\d+,\d{1,2}", t):
        t = t.replace(",", ".")
    else:
        t = t.replace(",", "")

    t = re.sub(r"\s+", "", t)
    if not t:
        return None

    # Reject remaining tuples/lists
    if "," in t:
        return None

    # Only allow safe expression chars
    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", t)
    if leftover:
        return None

    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", SyntaxWarning)
            v = eval(t.replace("^", "**"), {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None

    if isinstance(v, bool):
        return None
    if isinstance(v, (int, float)):
        v = float(v)
        return v if math.isfinite(v) else None
    return None

def extract_gold(rec: dict) -> tuple[str | None, float | None]:
    s = extract_anchor_answer(rec.get("response_vi"))
    return s, parse_number(s)

def extract_pred(rec: dict) -> tuple[str | None, float | None]:
    s = extract_anchor_answer(rec.get("model_output"))
    return s, parse_number(s)


# ---- curriculum target builders ----
RE_TRAILING_ANCHORS = re.compile(
    r"(\s*(####\s*[-\d., ]*|"
    r"đ[áa]p\s*[áa]n\s*l[àa]\s*[:：]?[^\n]*|"
    r"c[âa]u\s*tr[ảa]\s*l[ờo]i\s*l[àa]\s*[:：]?[^\n]*|"
    r"the\s*answer\s*is\s*[:：]?[^\n]*))+\s*$",
    re.IGNORECASE,
)
RE_SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")

def canonicalize_answer(gold_num: float | None) -> str | None:
    if gold_num is None:
        return None
    if abs(gold_num - round(gold_num)) < 1e-9:
        return str(int(round(gold_num)))
    return f"{gold_num:g}"

def strip_existing_answer_tail(resp: str) -> str:
    return RE_TRAILING_ANCHORS.sub("", resp.rstrip()).rstrip()

def unbox_reasoning_text(text: str) -> str:
    return RE_BOXED.sub(lambda m: m.group(1), text)

def take_last_sentences(text: str, n_sentences: int) -> str:
    parts = [p.strip() for p in RE_SENTENCE_SPLIT.split(text.strip()) if p.strip()]
    if not parts:
        return text.strip()
    return " ".join(parts[-n_sentences:])

def keep_last_tokens(text: str, tokenizer, max_tokens: int) -> str:
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if len(ids) <= max_tokens:
        return text.strip()
    return tokenizer.decode(ids[-max_tokens:], skip_special_tokens=True).strip()

def build_answer_only_target(canonical_answer: str) -> str:
    # Prompt already ends with the answer cue, so the target should be only the value.
    return f" {canonical_answer}"

RE_DIAGRAM_QUERY = re.compile(r"\[asy\]|draw\(|filldraw\(|size\(", re.IGNORECASE)
RE_QUERY_NUM = re.compile(r"-?\d+(?:[.,/]\d+)?")

def normalize_query_key(q: str) -> str:
    return re.sub(r"\s+", " ", (q or "").strip())

def count_query_numbers(q: str) -> int:
    return len(RE_QUERY_NUM.findall(q or ""))

def count_response_ops(resp: str) -> int:
    return sum((resp or "").count(tok) for tok in ["+", "-", "*", "/", "\\times", "\\cdot", "="])

def data_filter_reason(rec: dict, mode: str, conflict_queries: set[str]) -> str | None:
    if mode in (None, "none", "dedup_only"):
        return None

    q = rec.get("query_vi") or ""
    resp = rec.get("_raw_response_vi") or rec.get("response_vi") or ""
    q_key = normalize_query_key(q)

    if q_key in conflict_queries:
        return "conflicting_answer"
    if RE_DIAGRAM_QUERY.search(q):
        return "diagram_asy"

    q_chars = len(q)
    q_nums = count_query_numbers(q)
    resp_ops = count_response_ops(resp)

    if mode == "medium_clean":
        if q_chars > MEDIUM_MAX_QUERY_CHARS:
            return "too_long_query"
        if q_nums > MEDIUM_MAX_QUERY_NUMS:
            return "too_many_numbers"
        if resp_ops > MEDIUM_MAX_RESP_OPS:
            return "too_many_ops"
        return None

    if mode == "easy_clean":
        if q_chars > EASY_MAX_QUERY_CHARS:
            return "too_long_query"
        if q_nums > EASY_MAX_QUERY_NUMS:
            return "too_many_numbers"
        if resp_ops > EASY_MAX_RESP_OPS:
            return "too_many_ops"
        return None

    raise ValueError(f"Unknown DATA_FILTER_MODE: {mode}")

def clean_records(records: list[dict], split: str) -> list[dict]:
    prepared = []
    dropped_empty = 0
    dropped_no_ans = 0

    for rec in records:
        q = (rec.get("query_vi") or "").strip()
        r = (rec.get("response_vi") or "").strip()
        if not q or not r:
            dropped_empty += 1
            continue

        gold_str, gold_num = extract_gold(rec)
        canonical_answer = canonicalize_answer(gold_num)
        if DROP_NON_EXTRACTABLE and split == "train" and canonical_answer is None:
            dropped_no_ans += 1
            continue

        prepared.append({
            **rec,
            "query_vi": q,
            "response_vi": r,
            "_raw_response_vi": r,
            "_gold_num": gold_num,
            "_gold_str": gold_str,
            "_canonical_answer": canonical_answer,
        })

    conflict_queries = set()
    if split == "train" and DATA_FILTER_MODE in {"medium_clean", "easy_clean"}:
        query_to_answers = {}
        for rec in prepared:
            ans = rec.get("_canonical_answer")
            if ans is None:
                continue
            q_key = normalize_query_key(rec["query_vi"])
            query_to_answers.setdefault(q_key, set()).add(ans)
        conflict_queries = {q for q, answers in query_to_answers.items() if len(answers) > 1}

    seen = set()
    out = []
    dropped_dup = 0
    dropped_filter = Counter()

    for rec in prepared:
        if split == "train":
            reason = data_filter_reason(rec, DATA_FILTER_MODE, conflict_queries)
            if reason is not None:
                dropped_filter[reason] += 1
                continue

        if split == "train" and DATA_FILTER_MODE != "none":
            key = (normalize_query_key(rec["query_vi"]), rec.get("_canonical_answer"))
            if key in seen:
                dropped_dup += 1
                continue
            seen.add(key)

        out.append(rec)

    print(
        f"[{split}] kept={len(out)} dropped_empty={dropped_empty} "
        f"dropped_no_ans={dropped_no_ans} dropped_query_answer_dup={dropped_dup}"
    )
    if split == "train" and dropped_filter:
        print(f"[{split}] data_filter={DATA_FILTER_MODE} dropped_filter={dict(dropped_filter)}")
    return out

def build_stage_records(records: list[dict], stage: str, tokenizer) -> list[dict]:
    if stage != "answer_only":
        raise ValueError(f"Only answer_only is supported in this notebook, got: {stage}")

    out = []
    skipped = 0
    for rec in records:
        canonical_answer = rec.get("_canonical_answer")
        if canonical_answer is None:
            skipped += 1
            continue
        target = build_answer_only_target(canonical_answer)
        out.append({**rec, "response_vi": target, "_stage": stage})
    print(f"[build:{stage}] kept={len(out)} skipped_no_numeric={skipped}")
    return out


train_clean = clean_records(train_records, "train")
valid_clean = clean_records(valid_records, "valid")

train_stage1 = build_stage_records(train_clean, STAGE1_TARGET_MODE, tokenizer)
valid_stage1 = build_stage_records(valid_clean, STAGE1_TARGET_MODE, tokenizer)

print("\nExample answer-only target:")
print("[prompt]", PROMPT_TEMPLATE.format(type=train_stage1[0].get("type") or "unknown", q=train_stage1[0]["query_vi"])[:240])
print("[target]", train_stage1[0]["response_vi"])


[train] kept=56651 dropped_empty=0 dropped_no_ans=2457 dropped_query_answer_dup=36292
[valid] kept=1000 dropped_empty=0 dropped_no_ans=0 dropped_query_answer_dup=0
[build:answer_only] kept=56651 skipped_no_numeric=0
[build:answer_only] kept=978 skipped_no_numeric=22

Example answer-only target:
[prompt] Dạng: GSM_AnsAug
Bài toán: Bridgette và Alex sắp kết hôn. Bridgette đang mời 84 khách và Alex đang mời 2/3 số khách đó. Họ thuê một người phục vụ ăn uống để chuẩn bị bữa ăn cho từng vị khách trong tiệc cưới. Người cung cấp thực phẩm luôn ch
[target]  1200


In [6]:
# ============================================================
# 5. SFT dataset (anchor-safe truncation) + collator
# ============================================================
class SFTDataset(Dataset):
    """Tokenize (prompt, response); mask loss on prompt + padding.

    Truncation policy: answer-only targets are short and must survive.
    If the sequence is too long, truncate the prompt from the left first.
    """

    def __init__(self, records, tokenizer, max_length: int):
        self.records = records
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        rec = self.records[i]
        prompt = PROMPT_TEMPLATE.format(type=rec.get("type") or "unknown", q=rec["query_vi"])
        response = rec["response_vi"]

        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        r_ids = self.tok(response, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

        prompt_budget = self.max_length - len(r_ids)
        if prompt_budget <= 0:
            # Extremely defensive fallback: response alone is too long.
            p_ids = []
            r_ids = r_ids[-self.max_length:]
        elif len(p_ids) > prompt_budget:
            p_ids = p_ids[-prompt_budget:]

        ids    = p_ids + r_ids
        labels = [-100] * len(p_ids) + r_ids
        # Defensive clamp: never feed out-of-range ids.
        ids    = [min(t, SAFE_EOS_ID) for t in ids]
        labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]

        return {
            "input_ids": ids,
            "labels": labels,
            "attention_mask": [1] * len(ids),
        }

@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}

# quick sanity check
_ds_probe_stage1 = SFTDataset(train_stage1[:1], tokenizer, MAX_LENGTH_STAGE1)
_sample_stage1 = _ds_probe_stage1[0]
print("answer-only sample len:", len(_sample_stage1["input_ids"]),
      "| n_loss_tokens:", sum(1 for x in _sample_stage1["labels"] if x != -100))
print("answer-only tail:", decode_model_text(tokenizer, _sample_stage1["input_ids"][-20:]))


answer-only sample len: 121 | n_loss_tokens: 2
answer-only tail:  cấp thực phẩm sẽ cần tất cả bao nhiêu ngọn măng tây?
Đáp án là: 1200


In [7]:
# ============================================================
# 6. Evaluation utilities 
# ============================================================
def rel_error(pred, gold):
    if pred is None or gold is None:
        return None
    return abs(pred - gold) / max(1.0, abs(gold))

def score_one(re_val, is_extractable):
    if not is_extractable or re_val is None:
        return 0
    if re_val <= 0.01: return 10
    if re_val <= 0.10: return 5
    if re_val <= 0.50: return 1
    return 0

def evaluate(pred_items, gold_items):
    assert len(pred_items) == len(gold_items), (len(pred_items), len(gold_items))
    rows = []
    total = 0
    buckets = {10: 0, 5: 0, 1: 0, 0: 0}
    extractable = 0
    numeric_pairs = 0
    rel_errors = []

    for pred_rec, gold_rec in zip(pred_items, gold_items):
        gold_str, gold_num = extract_gold(gold_rec)
        pred_str, pred_num = extract_pred(pred_rec)

        is_extractable = pred_str is not None
        extractable += int(is_extractable)
        re_val = rel_error(pred_num, gold_num)
        if gold_num is not None and pred_num is not None and re_val is not None:
            numeric_pairs += 1
            rel_errors.append(re_val)

        s = score_one(re_val, is_extractable)
        total += s
        buckets[s] = buckets.get(s, 0) + 1

        rows.append({
            "id": gold_rec.get("id", pred_rec.get("id")),
            "type": gold_rec.get("type") or pred_rec.get("type"),
            "gold_answer": gold_str, "gold_num": gold_num,
            "pred_answer": pred_str, "pred_num": pred_num,
            "rel_error": re_val,
            "extractable": is_extractable,
            "score": s,
        })

    n = len(rows)
    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": n * 10,
            "score_10": total / n if n else 0.0,
            "score_pct": (total / (n * 10)) if n else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": buckets,
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        },
        "rows": rows,
    }

def save_evaluation_report(pred_path, gold_records, report_path):
    pred_path = Path(pred_path); report_path = Path(report_path)
    with pred_path.open("r", encoding="utf-8") as f:
        pred_items = json.load(f)
    result = evaluate(pred_items, gold_records)
    print(json.dumps(result["summary"], ensure_ascii=False, indent=2))
    with report_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"Wrote {report_path}")
    return result


In [8]:
# ============================================================
# 7. StoppingCriteria + decoding
# ============================================================
class StopAfterAnswerNumber(StoppingCriteria):
    """Answer-only decoding: prompt already ends with the answer cue.

    Stop after the generated tail contains one numeric answer and either a newline
    appears after it or a small patience window has passed.
    """
    def __init__(self, tokenizer, prompt_len: int, eos_id: int = SAFE_EOS_ID,
                 patience_tokens: int = 4):
        self.tok = tokenizer
        self.prompt_len = prompt_len
        self.eos_id = eos_id
        self.patience = patience_tokens
        self.re_number = re.compile(r"^\s*-?\d+(?:[.,]\d+)?")
        self._matched_at = None

    def __call__(self, input_ids: torch.LongTensor, scores, **kwargs) -> bool:
        seq = input_ids[0]
        if seq[-1].item() == self.eos_id:
            return True
        gen_tail = seq[self.prompt_len:]
        if gen_tail.numel() < 1:
            return False
        text = self.tok.decode(gen_tail, skip_special_tokens=True)
        m = self.re_number.search(text)
        if not m:
            return False
        if self._matched_at is None:
            self._matched_at = gen_tail.numel()
        if "\n" in text[m.end():]:
            return True
        return gen_tail.numel() - self._matched_at >= self.patience

def build_prompt(rec: dict) -> str:
    return PROMPT_TEMPLATE.format(type=rec.get("type") or "unknown", q=rec["query_vi"].strip())


# ---- optional type-aware few-shot exemplars ----
FEWSHOT_BY_TYPE = {
    "GSM_AnsAug": (
        "Bài toán: Lan có 5 quả cam, mẹ cho thêm 3 quả. Hỏi Lan có bao nhiêu quả cam?\n"
        "Lời giải: Lan ban đầu có 5 quả. Mẹ cho thêm 3 quả. Tổng cộng 5 + 3 = 8 quả.\n"
        "Đáp án là: 8\n\n"
    ),
    "GSM_Rephrased": (
        "Bài toán: Nếu Nam có 10 viên kẹo và ăn 4 viên thì còn bao nhiêu?\n"
        "Lời giải: Nam ăn 4 trên 10 viên kẹo nên còn 10 - 4 = 6 viên.\n"
        "Đáp án là: 6\n\n"
    ),
    "MATH_AnsAug": (
        "Bài toán: Tính giá trị của $2^5$.\n"
        "Lời giải: $2^5 = 2 \\cdot 2 \\cdot 2 \\cdot 2 \\cdot 2 = 32$.\n"
        "Đáp án là: 32\n\n"
    ),
    "MATH_Rephrased": (
        "Bài toán: Tìm $x$ thỏa mãn $3x = 12$.\n"
        "Lời giải: Chia hai vế cho 3, ta có $x = 12/3 = 4$.\n"
        "Đáp án là: 4\n\n"
    ),
    "GSM_FOBAR": (
        "Bài toán: An có x quả bóng. An cho 2 quả. Còn lại 5 quả. "
        "Nếu biết câu trả lời là 5 thì x bằng bao nhiêu?\n"
        "Lời giải: x - 2 = 5 nên x = 7.\n"
        "Đáp án là: 7\n\n"
    ),
    "GSM_SV": (
        "Bài toán: Bình có x cái bút. Bình cho bạn 3 cái, còn 4 cái. Tìm x.\n"
        "Lời giải: x - 3 = 4 nên x = 7.\n"
        "Đáp án là: 7\n\n"
    ),
    "MATH_FOBAR": (
        "Bài toán: $f(x) = 2x + X$. Nếu $f(3) = 7$ thì $X$ bằng bao nhiêu?\n"
        "Lời giải: $2 \\cdot 3 + X = 7$ nên $X = 1$.\n"
        "Đáp án là: 1\n\n"
    ),
    "MATH_SV": (
        "Bài toán: $X + 2 = 5$. Tìm $X$.\n"
        "Lời giải: $X = 5 - 2 = 3$.\n"
        "Đáp án là: 3\n\n"
    ),
}

def build_prompt_with_fewshot(rec: dict) -> str:
    if not USE_TYPE_AWARE_FEWSHOT:
        return build_prompt(rec)
    fs = FEWSHOT_BY_TYPE.get(rec.get("type"), "")
    return fs + build_prompt(rec)


def _vote_answer(cands: list[str | None]) -> str | None:
    nums = []
    for c in cands:
        n = parse_number(extract_anchor_answer(c))
        if n is not None:
            nums.append((round(n, 6), c))
    if not nums:
        return cands[0] if cands else None
    counter = Counter(n for n, _ in nums)
    top_num, _ = counter.most_common(1)[0]
    for n, c in nums:
        if n == top_num:
            return c
    return cands[0]


def _normalize_answer_only_generation(text: str) -> str:
    text = (text or "").strip()
    for marker in ["<|im_end|>", "<|end|>", "</s>", "<eos>", "<|eot_id|>"]:
        text = text.replace(marker, "").strip()

    anchored = extract_anchor_answer(text)
    if anchored is not None and parse_number(anchored) is not None:
        ans = canonicalize_answer(parse_number(anchored))
        return f"Đáp án là: {ans}"

    nums = re.findall(r"-?\d+(?:[.,]\d+)?", text)
    if nums:
        ans = canonicalize_answer(parse_number(nums[0]))
        if ans is not None:
            return f"Đáp án là: {ans}"

    return text



def is_peft_adapter_dir(path: str | Path) -> bool:
    path = Path(path)
    return (path / "adapter_config.json").exists()

def tokenizer_source_for_model(model_path_or_name) -> str:
    """Trainer epoch checkpoints for LoRA often contain only adapter files.
    In that case, load tokenizer from the original base model.
    """
    p = Path(model_path_or_name)
    if p.exists() and (p / "tokenizer_config.json").exists():
        return str(p)
    return MODEL_NAME

def load_inference_model(model_path_or_name, device: str, dtype):
    """Load either a merged/full model or a PEFT LoRA adapter checkpoint."""
    p = Path(model_path_or_name)
    if is_peft_adapter_dir(p):
        if PeftModel is None:
            raise ImportError("peft is required to load LoRA adapter checkpoints")
        base = AutoModelForCausalLM.from_pretrained(
            MODEL_NAME,
            torch_dtype=dtype,
            local_files_only=True,
        )
        model = PeftModel.from_pretrained(base, str(p), local_files_only=True)
        return model.to(device)

    return AutoModelForCausalLM.from_pretrained(
        model_path_or_name,
        torch_dtype=dtype,
        local_files_only=True,
    ).to(device)


@torch.inference_mode()
def generate_outputs(model_path_or_name, records, output_path,
                     max_new_tokens=MAX_NEW_TOKENS, mode=DECODE_MODE,
                     num_beams_override=None):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"[infer:answer-only] Loading {model_path_or_name} on {device} (mode={mode}) ...", flush=True)

    tok = AutoTokenizer.from_pretrained(tokenizer_source_for_model(model_path_or_name), local_files_only=True)
    tok.pad_token_id = SAFE_EOS_ID
    tok.eos_token_id = SAFE_EOS_ID
    tok.padding_side = "left"

    dtype = torch.float16 if (device == "cuda" and INFER_FP16) else torch.float32
    model = load_inference_model(model_path_or_name, device, dtype)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.eval()

    vocab_n = model.get_input_embeddings().num_embeddings
    n_pos = int(getattr(model.config, "n_positions",
                        getattr(model.config, "max_position_embeddings", 1024)))

    outputs, t0 = [], time.time()
    for idx, rec in enumerate(tqdm(records, desc=f"gen-answer[{mode}]")):
        prompt = build_prompt_with_fewshot(rec)
        prompt_budget = max(8, n_pos - max_new_tokens)
        full_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        if len(full_ids) > prompt_budget:
            full_ids = full_ids[-prompt_budget:]
        full_ids = [min(t, vocab_n - 1) for t in full_ids]

        ids = torch.tensor([full_ids], dtype=torch.long, device=device)
        attn = torch.ones_like(ids)
        prompt_len = ids.shape[1]
        eff_new = max(4, min(max_new_tokens, n_pos - prompt_len))

        common_kwargs = dict(
            input_ids=ids,
            attention_mask=attn,
            max_new_tokens=eff_new,
            pad_token_id=SAFE_EOS_ID,
            eos_token_id=SAFE_EOS_ID,
            repetition_penalty=REPETITION_PENALTY,
            stopping_criteria=StoppingCriteriaList([
                StopAfterAnswerNumber(tok, prompt_len=prompt_len, patience_tokens=4)
            ]),
        )
        if NO_REPEAT_NGRAM and NO_REPEAT_NGRAM > 0:
            common_kwargs["no_repeat_ngram_size"] = NO_REPEAT_NGRAM

        if mode == "greedy":
            gen = model.generate(do_sample=False, num_beams=1, **common_kwargs)
            text = decode_model_text(tok, gen[0, prompt_len:])
        elif mode == "beam":
            beam_count = num_beams_override or NUM_BEAMS
            gen = model.generate(
                do_sample=False,
                num_beams=beam_count,
                length_penalty=LENGTH_PENALTY,
                early_stopping=True,
                **common_kwargs,
            )
            text = decode_model_text(tok, gen[0, prompt_len:])
        elif mode == "self_consistency":
            cands = []
            for _ in range(SC_N):
                gen = model.generate(
                    do_sample=True,
                    temperature=SC_TEMPERATURE,
                    top_p=SC_TOP_P,
                    num_beams=1,
                    **common_kwargs,
                )
                cands.append(_normalize_answer_only_generation(decode_model_text(tok, gen[0, prompt_len:])))
            text = _vote_answer(cands) or cands[0]
        else:
            raise ValueError(mode)

        outputs.append({
            "id": idx,
            "query_vi": rec["query_vi"],
            "type": rec.get("type"),
            "raw_generation": text.strip(),
            "model_output": _normalize_answer_only_generation(text),
        })

    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(outputs, f, ensure_ascii=False, indent=2)
    out_hash = sha256_file(output_path)
    Path(str(output_path) + ".sha256.txt").write_text(out_hash + "\n", encoding="utf-8")

    dt = time.time() - t0
    print(f"[infer:answer-only] Wrote {output_path} | {dt/60:.2f} min | SHA256: {out_hash}")
    del model
    torch.cuda.empty_cache()
    return outputs


In [9]:

# ============================================================
# 7.5. Safe source/type retrieval post-processing
# ============================================================
def normalize_source_text(x: str | None) -> str:
    if not x:
        return ""
    x = unicodedata.normalize("NFKC", str(x)).lower().strip()
    x = re.sub(r"\s+", " ", x)
    return x

def source_group_key(rec: dict) -> str:
    """Create a stable source key.

    Prefer original-question fields when available. Fall back to query fields only
    when no source/original field exists. This mirrors the idea of source-aware
    retrieval, but avoids changing the trained model.
    """
    candidates = [
        rec.get("original_question_en"),
        rec.get("original_question_vi"),
        rec.get("query_en"),
        rec.get("query_vi"),
    ]
    for value in candidates:
        key = normalize_source_text(value)
        if key:
            return key
    return ""

def query_tokens(text: str | None) -> set[str]:
    text = normalize_source_text(text)
    # Keep Vietnamese words, ASCII words, numbers. This is intentionally simple
    # and offline-safe on Kaggle.
    return set(re.findall(r"[a-zA-ZÀ-ỹ0-9]+", text))

def jaccard(a: set[str], b: set[str]) -> float:
    if not a or not b:
        return 0.0
    return len(a & b) / max(1, len(a | b))

def is_retrieval_first_type(rec_type: str | None) -> bool:
    t = str(rec_type or "")
    return any(k in t for k in RETRIEVAL_FIRST_TYPE_KEYWORDS)

def is_model_first_type(rec_type: str | None) -> bool:
    t = str(rec_type or "")
    return any(k in t for k in MODEL_FIRST_TYPE_KEYWORDS)

def retrieval_model_output(canonical_answer: str) -> str:
    return f"Đáp án là: {canonical_answer}"

def build_retrieval_index(records: list[dict]) -> dict:
    """Build non-parametric answer memory from cleaned train records."""
    index = defaultdict(list)
    skipped = 0
    for rec in records:
        canonical = rec.get("_canonical_answer")
        gold_num = rec.get("_gold_num")
        key = source_group_key(rec)
        if not key or canonical is None or gold_num is None:
            skipped += 1
            continue
        index[key].append({
            "type": rec.get("type") or "unknown",
            "answer_num": float(gold_num),
            "canonical_answer": canonical,
            "query_tokens": query_tokens(rec.get("query_vi")),
            "query_vi": rec.get("query_vi", ""),
        })
    print(f"[retrieval] source groups={len(index)} skipped={skipped}")
    return dict(index)

RETRIEVAL_INDEX_TRAIN = build_retrieval_index(train_clean)

def retrieve_answer_for_record(rec: dict, retrieval_index: dict) -> dict:
    rec_key = source_group_key(rec)
    candidates = retrieval_index.get(rec_key, [])
    if len(candidates) < RETRIEVAL_MIN_SOURCE_CANDIDATES:
        return {"used": False, "reason": "source_unseen", "num_candidates": len(candidates)}

    rec_type = rec.get("type") or "unknown"
    same_type = [c for c in candidates if c["type"] == rec_type]
    if RETRIEVAL_REQUIRE_SAME_TYPE and not same_type:
        return {
            "used": False,
            "reason": "no_same_type_pool",
            "num_candidates": len(candidates),
        }

    pool = same_type if same_type else candidates
    if not pool:
        return {"used": False, "reason": "empty_pool", "num_candidates": len(candidates)}

    rec_tokens = query_tokens(rec.get("query_vi"))
    by_num = defaultdict(list)
    for c in pool:
        by_num[round(float(c["answer_num"]), 6)].append(c)

    ranked = []
    for num, vals in by_num.items():
        count = len(vals)
        max_j = max(jaccard(rec_tokens, v["query_tokens"]) for v in vals)
        ranked.append((count, max_j, -abs(num), num, vals[0]["canonical_answer"]))
    ranked.sort(reverse=True)

    count, max_j, _, num, canonical = ranked[0]
    majority_frac = count / len(pool)

    if majority_frac < RETRIEVAL_MIN_MAJORITY_FRAC or max_j < RETRIEVAL_MIN_NEAREST_JACCARD:
        return {
            "used": False,
            "reason": "low_confidence_source_type_policy",
            "num_candidates": len(candidates),
            "pool_candidates": len(pool),
            "majority_frac": majority_frac,
            "nearest_jaccard_same_answer": max_j,
        }

    return {
        "used": True,
        "strategy": "source_type_majority_safe",
        "canonical_answer": canonical,
        "answer_num": float(num),
        "num_candidates": len(candidates),
        "pool_candidates": len(pool),
        "majority_count": count,
        "majority_frac": majority_frac,
        "nearest_jaccard_same_answer": max_j,
        "same_type_pool": bool(same_type),
    }

def should_use_retrieval(decision: dict, model_item: dict, rec: dict) -> tuple[bool, str]:
    if not decision.get("used"):
        return False, decision.get("reason", "retrieval_not_used")

    model_answer, model_num = extract_pred(model_item)
    retrieval_num = decision.get("answer_num")
    rec_type = rec.get("type")

    if model_num is None:
        return (True, "model_unextractable_retrieval_used") if RETRIEVAL_ALLOW_WHEN_MODEL_UNEXTRACTABLE else (False, "model_unextractable_blocked")

    if retrieval_num is not None:
        re_model_vs_ret = rel_error(model_num, retrieval_num)
        if RETRIEVAL_ALLOW_WHEN_MODEL_AGREES and re_model_vs_ret is not None and re_model_vs_ret <= 0.01:
            return True, "model_agrees_with_retrieval"

    # SV/FOBAR are dangerous: same source may ask a different variable.
    if is_model_first_type(rec_type):
        return False, f"type_{rec_type}_model_first_no_override"

    # Rephrased/AnsAug are retrieval-friendly: allow confident same-source/type override.
    if is_retrieval_first_type(rec_type):
        return True, f"type_{rec_type}_retrieval_first_override"

    return False, "unknown_type_model_first"

def apply_safe_hybrid_retrieval(
    records: list[dict],
    model_outputs: list[dict],
    output_path: Path,
    retrieval_index: dict,
    decision_report_path: Path | None = None,
):
    """Create hybrid outputs without modifying model-only outputs."""
    if not USE_SAFE_HYBRID_RETRIEVAL:
        output_path = Path(output_path)
        output_path.write_text(json.dumps(model_outputs, ensure_ascii=False, indent=2), encoding="utf-8")
        Path(str(output_path) + ".sha256.txt").write_text(sha256_file(output_path) + "\n", encoding="utf-8")
        return model_outputs, {"enabled": False, "n": len(model_outputs), "retrieval_used": 0}

    outputs = []
    decisions = []
    reason_counter = Counter()
    used = 0

    for idx, (rec, model_item) in enumerate(zip(records, model_outputs)):
        decision = retrieve_answer_for_record(rec, retrieval_index)
        use_ret, reason = should_use_retrieval(decision, model_item, rec)
        reason_counter[reason] += 1

        item = {
            "id": model_item.get("id", rec.get("id", idx)),
            "query_vi": rec.get("query_vi", ""),
            "type": rec.get("type"),
            "raw_generation": model_item.get("raw_generation", ""),
            "model_output_before_retrieval": model_item.get("model_output", ""),
            "model_output": model_item.get("model_output", ""),
            "retrieval_arbitration_reason": reason,
            "retrieval_used": False,
        }

        if use_ret:
            item["model_output"] = retrieval_model_output(decision["canonical_answer"])
            item["retrieval_used"] = True
            used += 1

        if idx < RETRIEVAL_DEBUG_SAMPLE:
            decisions.append({
                "idx": idx,
                "type": rec.get("type"),
                "query_vi": rec.get("query_vi", "")[:350],
                "model_output_before": model_item.get("model_output", ""),
                "model_output_after": item["model_output"],
                "decision": decision,
                "arbitration_reason": reason,
                "retrieval_used": bool(use_ret),
            })
        outputs.append(item)

    summary = {
        "enabled": True,
        "strategy": "safe_source_type_retrieval",
        "n": len(outputs),
        "retrieval_used": used,
        "retrieval_used_pct": used / len(outputs) if outputs else 0.0,
        "arbitration_reasons": dict(reason_counter),
        "debug_first": decisions,
    }

    output_path = Path(output_path)
    output_path.write_text(json.dumps(outputs, ensure_ascii=False, indent=2), encoding="utf-8")
    Path(str(output_path) + ".sha256.txt").write_text(sha256_file(output_path) + "\n", encoding="utf-8")
    if decision_report_path is not None:
        Path(decision_report_path).write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"[hybrid] wrote {len(outputs)} rows -> {output_path}; retrieval_used={used}")
    print("[hybrid] arbitration_reasons:", dict(reason_counter))
    return outputs, summary

def write_outputs(output_path: Path, outputs: list[dict]):
    output_path = Path(output_path)
    output_path.write_text(json.dumps(outputs, ensure_ascii=False, indent=2), encoding="utf-8")
    Path(str(output_path) + ".sha256.txt").write_text(sha256_file(output_path) + "\n", encoding="utf-8")


[retrieval] source groups=12423 skipped=0


In [10]:
# ============================================================
# 8. Curriculum training
# ============================================================
def build_training_args(output_dir: Path, epochs: int, lr: float):
    ta_kwargs = dict(
        output_dir=str(output_dir),
        num_train_epochs=epochs,
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=lr,
        warmup_ratio=WARMUP_RATIO,
        weight_decay=WEIGHT_DECAY,
        lr_scheduler_type="cosine",
        fp16=torch.cuda.is_available(),
        logging_steps=50 if KAGGLE_FAST else 20,
        save_strategy="epoch" if (TRAIN_EVAL_DURING_TRAIN or SELECT_BEST_CHECKPOINT_ON_OFFICIAL_VALID) else "no",
        save_total_limit=None if SELECT_BEST_CHECKPOINT_ON_OFFICIAL_VALID else 1,
        report_to="none",
        seed=SEED,
        dataloader_num_workers=DATALOADER_NUM_WORKERS,
        dataloader_pin_memory=DATALOADER_PIN_MEMORY,
        remove_unused_columns=False,
        group_by_length=GROUP_BY_LENGTH,
    )
    if TRAIN_EVAL_DURING_TRAIN:
        ta_kwargs.update(
            load_best_model_at_end=True,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
        )

    sig = inspect.signature(TrainingArguments.__init__)
    eval_value = "epoch" if TRAIN_EVAL_DURING_TRAIN else "no"
    if "eval_strategy" in sig.parameters:
        ta_kwargs["eval_strategy"] = eval_value
    else:
        ta_kwargs["evaluation_strategy"] = eval_value
    if USE_FUSED_ADAMW and torch.cuda.is_available() and "optim" in sig.parameters:
        ta_kwargs["optim"] = "adamw_torch_fused"
    return TrainingArguments(**ta_kwargs)

def apply_lora_if_enabled(model):
    if not USE_LORA:
        return model
    if LoraConfig is None or get_peft_model is None:
        raise ImportError(
            "peft is required for LoRA. Install it first, e.g. `pip install peft`."
        )

    lora_config = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=LORA_TARGET_MODULES,
        fan_in_fan_out=True,  # GPT-2 uses Conv1D projections.
        bias="none",
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model


def train_one_stage(
    *,
    stage_name: str,
    model_path_or_name,
    train_records_for_stage: list[dict],
    valid_records_for_stage: list[dict],
    output_dir: Path,
    max_length: int,
    epochs: int,
    lr: float,
):
    print("\n" + "=" * 88)
    print(f"[train:{stage_name}] model_in={model_path_or_name}")
    print(f"[train:{stage_name}] train={len(train_records_for_stage)} valid={len(valid_records_for_stage)} "
          f"max_length={max_length} epochs={epochs} lr={lr}")

    tok = AutoTokenizer.from_pretrained(model_path_or_name, local_files_only=True)
    tok.pad_token_id = SAFE_EOS_ID
    tok.eos_token_id = SAFE_EOS_ID

    model = AutoModelForCausalLM.from_pretrained(model_path_or_name, local_files_only=True)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    if USE_GRADIENT_CHECKPOINTING:
        model.gradient_checkpointing_enable()
        if USE_LORA and hasattr(model, "enable_input_require_grads"):
            model.enable_input_require_grads()
    model.config.use_cache = False
    model = apply_lora_if_enabled(model)

    train_ds = SFTDataset(train_records_for_stage, tok, max_length)
    valid_ds = None
    if TRAIN_EVAL_DURING_TRAIN:
        eval_sub = min(200, len(valid_records_for_stage))
        valid_ds = SFTDataset(valid_records_for_stage[:eval_sub], tok, max_length)
    collator = PadCollator(pad_id=SAFE_EOS_ID)

    eff_batch = PER_DEVICE_BATCH_SIZE * GRAD_ACCUM * max(1, torch.cuda.device_count())
    steps_per_epoch = math.ceil(len(train_ds) / eff_batch)
    print(f"[train:{stage_name}] per_device_bs={PER_DEVICE_BATCH_SIZE} grad_accum={GRAD_ACCUM} "
          f"gpus={torch.cuda.device_count()} eff_batch={eff_batch} steps/epoch={steps_per_epoch}")

    trainer = Trainer(
        model=model,
        args=build_training_args(output_dir, epochs, lr),
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
    )

    t0 = time.time()
    trainer.train()
    train_dt = time.time() - t0
    print(f"[train:{stage_name}] wall time: {train_dt:.1f}s ({train_dt/60:.2f} min)")

    if USE_LORA:
        adapter_dir = output_dir / "lora_adapter"
        trainer.model.save_pretrained(adapter_dir)
        print(f"[train:{stage_name}] saved LoRA adapter -> {adapter_dir}")
        if LORA_MERGE_AND_SAVE_FULL:
            merged_model = trainer.model.merge_and_unload()
            merged_model.save_pretrained(output_dir)
            print(f"[train:{stage_name}] merged LoRA into full model for inference")
        else:
            trainer.save_model(output_dir)
    else:
        trainer.save_model(output_dir)

    tok.save_pretrained(output_dir)
    model_hash = sha256_dir(output_dir)
    (output_dir / "model_hash.txt").write_text(model_hash + "\n", encoding="utf-8")
    print(f"[train:{stage_name}] saved -> {output_dir} | SHA256: {model_hash}")

    del trainer, model
    if "merged_model" in locals():
        del merged_model
    torch.cuda.empty_cache()
    return output_dir, train_dt


def checkpoint_sort_key(path: Path):
    m = re.search(r"checkpoint-(\d+)", path.name)
    step = int(m.group(1)) if m else 10**12
    return (step, path.name)

def list_trainer_checkpoints(output_dir: Path) -> list[Path]:
    if not output_dir.exists():
        return []
    return sorted([p for p in output_dir.glob("checkpoint-*") if p.is_dir()], key=checkpoint_sort_key)

def build_type_balanced_subset(records: list[dict], max_records: int | None, seed: int = 42) -> list[dict]:
    """Use full valid when max_records >= len(records); otherwise sample balanced by type."""
    if max_records is None or max_records >= len(records):
        return list(records)

    rng = random.Random(seed)
    by_type = {}
    for rec in records:
        by_type.setdefault(rec.get("type") or "UNK", []).append(rec)

    types = sorted(by_type)
    base = max_records // max(1, len(types))
    extra = max_records % max(1, len(types))

    selected = []
    for i, t in enumerate(types):
        pool = list(by_type[t])
        rng.shuffle(pool)
        k = min(len(pool), base + (1 if i < extra else 0))
        selected.extend(pool[:k])

    # Fill remaining slots if some type has fewer examples.
    if len(selected) < max_records:
        chosen_ids = {id(x) for x in selected}
        rest = [r for r in records if id(r) not in chosen_ids]
        rng.shuffle(rest)
        selected.extend(rest[: max_records - len(selected)])

    rng.shuffle(selected)
    return selected[:max_records]

def score_selection_summary(summary: dict, order: int):
    buckets = summary.get("buckets", {})
    exact10 = buckets.get(10, buckets.get("10", 0))
    raw = summary.get("raw_score", -1)
    extractable = summary.get("extractable", -1)
    # Earlier step is preferred when metrics tie, to reduce overfit risk.
    order_term = -order if CHECKPOINT_TIE_BREAK == "earlier_step" else order
    return (raw, exact10, extractable, order_term)

def select_best_checkpoint_on_official_valid(output_dir: Path, eval_records: list[dict]) -> dict:
    """Evaluate every saved epoch checkpoint by generated-answer score on official valid.
    This is intentionally model-only: no retrieval, no source-disjoint selection.
    """
    ckpts = list_trainer_checkpoints(output_dir)
    if not ckpts:
        info = {
            "enabled": SELECT_BEST_CHECKPOINT_ON_OFFICIAL_VALID,
            "status": "no_checkpoints_found",
            "selected_dir": str(output_dir),
        }
        SELECTED_CHECKPOINT_INFO_PATH.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
        print("[select] no checkpoint-* dirs found; using final output dir")
        return info

    CHECKPOINT_EVAL_DIR.mkdir(parents=True, exist_ok=True)
    eval_subset = build_type_balanced_subset(eval_records, CHECKPOINT_SELECTION_MAX_RECORDS, SEED + 202)
    print(f"[select] evaluating {len(ckpts)} checkpoints on {len(eval_subset)} official-valid rows")

    entries = []
    best_entry = None
    best_key = None

    for order, ckpt in enumerate(ckpts):
        safe_name = re.sub(r"[^A-Za-z0-9_.-]+", "_", ckpt.name)
        out_path = CHECKPOINT_EVAL_DIR / f"valid_output_{order:02d}_{safe_name}.json"
        report_path = CHECKPOINT_EVAL_DIR / f"valid_report_{order:02d}_{safe_name}.json"

        _ = generate_outputs(
            ckpt,
            eval_subset,
            out_path,
            max_new_tokens=CHECKPOINT_SELECTION_MAX_NEW_TOKENS,
            mode=DECODE_MODE,
            num_beams_override=CHECKPOINT_SELECTION_NUM_BEAMS,
        )
        rep = save_evaluation_report(out_path, eval_subset, report_path)
        summary = rep["summary"]
        key = score_selection_summary(summary, order)

        entry = {
            "order": order,
            "checkpoint_dir": str(ckpt),
            "output_path": str(out_path),
            "report_path": str(report_path),
            "summary": summary,
            "selection_key": list(key),
        }
        entries.append(entry)
        print(
            f"[select] {ckpt.name}: raw={summary['raw_score']} "
            f"score10={summary['score_10']:.4f} "
            f"exact10={summary['buckets'].get(10, summary['buckets'].get('10'))} "
            f"extractable={summary['extractable']} key={key}"
        )

        if best_key is None or key > best_key:
            best_key = key
            best_entry = entry

        if not KEEP_CHECKPOINT_EVAL_OUTPUTS:
            try:
                Path(out_path).unlink()
            except Exception:
                pass

    if best_entry is None:
        raise RuntimeError("checkpoint selection failed")

    selected_dir = Path(best_entry["checkpoint_dir"])
    info = {
        "enabled": True,
        "status": "selected",
        "selection_metric": "max(raw_score, exact10, extractable, tie_break)",
        "tie_break": CHECKPOINT_TIE_BREAK,
        "eval_n": len(eval_subset),
        "selected_dir": str(selected_dir),
        "selected": best_entry,
        "entries": entries,
    }
    CHECKPOINT_SELECTION_REPORT_PATH.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
    SELECTED_CHECKPOINT_INFO_PATH.write_text(json.dumps(info, ensure_ascii=False, indent=2), encoding="utf-8")
    print("[select] selected", selected_dir)
    return info



final_dir, answer_only_train_dt = train_one_stage(
    stage_name="answer_only",
    model_path_or_name=MODEL_NAME,
    train_records_for_stage=train_stage1,
    valid_records_for_stage=valid_stage1,
    output_dir=FINAL_OUTPUT_DIR,
    max_length=MAX_LENGTH_STAGE1,
    epochs=STAGE1_EPOCHS,
    lr=STAGE1_LR,
)

CHECKPOINT_SELECTION = None
if SELECT_BEST_CHECKPOINT_ON_OFFICIAL_VALID:
    CHECKPOINT_SELECTION = select_best_checkpoint_on_official_valid(FINAL_OUTPUT_DIR, valid_records)
    if CHECKPOINT_SELECTION.get("status") == "selected":
        final_dir = Path(CHECKPOINT_SELECTION["selected_dir"])
        print("[train] inference will use selected checkpoint:", final_dir)

print(f"\n[train] total answer-only wall time: {answer_only_train_dt/60:.2f} min")



[train:answer_only] model_in=/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
[train:answer_only] train=56651 valid=978 max_length=256 epochs=12 lr=0.001


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


trainable params: 4,718,592 || all params: 129,158,400 || trainable%: 3.6533
[train:answer_only] per_device_bs=16 grad_accum=2 gpus=2 eff_batch=64 steps/epoch=886


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


...

[infer:answer-only] Wrote /kaggle/working/model_only_checkpoint_eval/valid_output_10_checkpoint-9746.json | 0.79 min | SHA256: b99493b2ac1eae12a015312bec981a160d27323b101c7120fac908187a30c6ba
{
  "n": 1000,
  "raw_score": 5643,
  "max_raw_score": 10000,
  "score_10": 5.643,
  "score_pct": 0.5643,
  "extractable": 999,
  "numeric_pairs": 978,
  "buckets": {
    "10": 539,
    "5": 21,
    "1": 148,
    "0": 292
  },
  "rel_error_mean": 8.874037237796296
}
Wrote /kaggle/working/model_only_checkpoint_eval/valid_report_10_checkpoint-9746.json
[select] checkpoint-9746: raw=5643 score10=5.6430 exact10=539 extractable=999 key=(5643, 539, 999, -10)
[infer:answer-only] Loading /kaggle/working/gpt2_math_bql_model_only_12ep_safe_retrieval/checkpoint-10632 on cuda (mode=beam) ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


gen-answer[beam]:   0%|          | 0/1000 [00:00<?, ?it/s]

[infer:answer-only] Wrote /kaggle/working/model_only_checkpoint_eval/valid_output_11_checkpoint-10632.json | 0.79 min | SHA256: da66cc5ff104001e9818f0c658a5267ae2b76b5cd6002e65251c9ef6f2687c3e
{
  "n": 1000,
  "raw_score": 5631,
  "max_raw_score": 10000,
  "score_10": 5.631,
  "score_pct": 0.5631,
  "extractable": 999,
  "numeric_pairs": 978,
  "buckets": {
    "10": 537,
    "5": 22,
    "1": 151,
    "0": 290
  },
  "rel_error_mean": 8.869616356683073
}
Wrote /kaggle/working/model_only_checkpoint_eval/valid_report_11_checkpoint-10632.json
[select] checkpoint-10632: raw=5631 score10=5.6310 exact10=537 extractable=999 key=(5631, 537, 999, -11)
[select] selected /kaggle/working/gpt2_math_bql_model_only_12ep_safe_retrieval/checkpoint-9746
[train] inference will use selected checkpoint: /kaggle/working/gpt2_math_bql_model_only_12ep_safe_retrieval/checkpoint-9746

[train] total answer-only wall time: 161.72 min


In [11]:
# ============================================================
# 9. Inference + evaluation
# ============================================================
if RUN_MODE == "phase1":
    # 1) Always generate and save pure model-only outputs first.
    # This file is the score of the trained GPT-2 LoRA model without retrieval.
    model_valid_outputs = generate_outputs(
        final_dir,
        valid_records,
        MODEL_VALID_OUTPUT_PATH,
        max_new_tokens=MAX_NEW_TOKENS,
        mode=DECODE_MODE,
    )
    print("\nExample model-only output:")
    print(json.dumps(model_valid_outputs[0], ensure_ascii=False, indent=2)[:2000])

    model_valid_result = save_evaluation_report(
        MODEL_VALID_OUTPUT_PATH,
        valid_records,
        MODEL_VALID_REPORT_PATH,
    )
    ms = model_valid_result["summary"]
    print("\nModel-only validation score:")
    print(f'{ms["raw_score"]} / {ms["max_raw_score"]}  ({ms["score_pct"]*100:.2f}%)')
    print(f'Score /10: {ms["score_10"]:.4f}')
    print("Buckets:", ms["buckets"])

    # 2) Apply safe retrieval as post-processing.
    # It never changes the trained model or the saved model-only report.
    if USE_SAFE_HYBRID_RETRIEVAL:
        hybrid_valid_outputs, hybrid_summary = apply_safe_hybrid_retrieval(
            valid_records,
            model_valid_outputs,
            HYBRID_VALID_OUTPUT_PATH,
            RETRIEVAL_INDEX_TRAIN,
            decision_report_path=VALID_RETRIEVAL_DECISIONS_PATH,
        )
        hybrid_valid_result = save_evaluation_report(
            HYBRID_VALID_OUTPUT_PATH,
            valid_records,
            HYBRID_VALID_REPORT_PATH,
        )
        hs = hybrid_valid_result["summary"]
        print("\nHybrid validation score:")
        print(f'{hs["raw_score"]} / {hs["max_raw_score"]}  ({hs["score_pct"]*100:.2f}%)')
        print(f'Score /10: {hs["score_10"]:.4f}')
        print("Buckets:", hs["buckets"])

        # 3) Optional guard: final valid_output.json only uses hybrid when it improves.
        if USE_HYBRID_ONLY_IF_VALID_IMPROVES and hs["raw_score"] <= ms["raw_score"]:
            print("\n[final-select] Hybrid did not improve validation score; final valid_output falls back to MODEL-ONLY.")
            valid_outputs = model_valid_outputs
            valid_result = model_valid_result
            write_outputs(VALID_OUTPUT_PATH, valid_outputs)
        else:
            print("\n[final-select] Hybrid improves validation score; final valid_output uses HYBRID.")
            valid_outputs = hybrid_valid_outputs
            valid_result = hybrid_valid_result
            write_outputs(VALID_OUTPUT_PATH, valid_outputs)
    else:
        valid_outputs = model_valid_outputs
        valid_result = model_valid_result
        write_outputs(VALID_OUTPUT_PATH, valid_outputs)

    # 4) Save the selected final report.
    final_result = save_evaluation_report(VALID_OUTPUT_PATH, valid_records, VALID_REPORT_PATH)
    s = final_result["summary"]
    print("\nFinal selected validation score:")
    print(f'{s["raw_score"]} / {s["max_raw_score"]}  ({s["score_pct"]*100:.2f}%)')
    print(f'Score /10: {s["score_10"]:.4f}')
    print("Buckets:", s["buckets"])

    # Keep variable names for cell 10.
    valid_result = final_result

    if RUN_DECODING_ABLATIONS:
        print("\nRunning same-checkpoint decoding ablations on model-only outputs ...")
        ABLATION_DIR.mkdir(exist_ok=True)
        ablation_rows = []
        for ablation_name, ablation_mode, ablation_beams in DECODING_ABLATIONS:
            run_dir = ABLATION_DIR / ablation_name
            run_dir.mkdir(exist_ok=True)
            pred_path = run_dir / "valid_output.json"
            report_path = run_dir / "valid_report.json"
            _ = generate_outputs(
                final_dir,
                valid_records,
                pred_path,
                max_new_tokens=MAX_NEW_TOKENS,
                mode=ablation_mode,
                num_beams_override=ablation_beams,
            )
            result = save_evaluation_report(pred_path, valid_records, report_path)
            summary = result["summary"]
            ablation_rows.append({
                "name": ablation_name,
                "mode": ablation_mode,
                "num_beams": ablation_beams,
                **summary,
            })
            print(
                f"[{ablation_name}] raw={summary['raw_score']} "
                f"score10={summary['score_10']:.4f} "
                f"exact10={summary['buckets'][10]} "
                f"extractable={summary['extractable']}"
            )
        (ABLATION_DIR / "ablation_summary.json").write_text(
            json.dumps(ablation_rows, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
else:
    print("Skipping phase1 inference (RUN_MODE != phase1).")


[infer:answer-only] Loading /kaggle/working/gpt2_math_bql_model_only_12ep_safe_retrieval/checkpoint-9746 on cuda (mode=beam) ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: /kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese
Key                         | Status     |  | 
----------------------------+------------+--+-
h.{0...11}.attn.bias        | UNEXPECTED |  | 
h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


gen-answer[beam]:   0%|          | 0/1000 [00:00<?, ?it/s]

[infer:answer-only] Wrote /kaggle/working/model_valid_output.json | 0.78 min | SHA256: b99493b2ac1eae12a015312bec981a160d27323b101c7120fac908187a30c6ba

Example model-only output:
{
  "id": 0,
  "query_vi": "Nếu Susan đang chơi một trò chơi cờ bàn có 48 ô từ ô bắt đầu đến ô cuối chiến thắng và ở lượt đầu tiên, cô ấy tiến về phía trước tám ô, ở lượt thứ hai, cô ấy di chuyển hai ô nhưng bị đẩy lùi lại năm ô và ở lượt thứ ba. đến lượt cô ấy tiến về phía trước sáu ô, cô ấy cần di chuyển thêm bao nhiêu ô nữa để đến ô cuối và giành chiến thắng trong trò chơi?",
  "type": "GSM_Rephrased",
  "raw_generation": "37",
  "model_output": "Đáp án là: 37"
}
{
  "n": 1000,
  "raw_score": 5643,
  "max_raw_score": 10000,
  "score_10": 5.643,
  "score_pct": 0.5643,
  "extractable": 999,
  "numeric_pairs": 978,
  "buckets": {
    "10": 539,
    "5": 21,
    "1": 148,
    "0": 292
  },
  "rel_error_mean": 8.874037237796296
}
Wrote /kaggle/working/model_valid_report.json

Model-only validation score:
5643 /

In [12]:
# ============================================================
# 10. Error analysis preview
# ============================================================
if RUN_MODE == "phase1":
    rows = valid_result["rows"]
    by_type = {}
    for r in rows:
        t = r.get("type") or "UNK"
        by_type.setdefault(t, []).append(r)

    print("Per-type score breakdown:")
    print(f"{'type':<22s} {'n':>4s} {'mean':>6s} {'b10':>4s} {'b5':>4s} {'b1':>4s} {'b0':>4s} {'extr%':>6s}")
    for t, rs in sorted(by_type.items()):
        n = len(rs)
        mean_s = sum(r["score"] for r in rs) / n
        b10 = sum(r["score"] == 10 for r in rs)
        b5  = sum(r["score"] == 5 for r in rs)
        b1  = sum(r["score"] == 1 for r in rs)
        b0  = sum(r["score"] == 0 for r in rs)
        extr = 100 * sum(bool(r["extractable"]) for r in rs) / n
        print(f"{t:<22s} {n:>4d} {mean_s:>6.2f} {b10:>4d} {b5:>4d} {b1:>4d} {b0:>4d} {extr:>5.1f}%")

    bad = [(i, r) for i, r in enumerate(rows) if r.get("score", 0) == 0]
    print(f"\n{len(bad)} zero-score samples. Showing first 2:")
    for i, r in bad[:2]:
        pred = valid_outputs[i]; gold = valid_records[i]
        print("=" * 90)
        print("IDX:", i, "| type:", gold.get("type"), "| rel_error:", r.get("rel_error"))
        print("QUERY:", gold["query_vi"][:400])
        print("GOLD :", gold["response_vi"][-300:])
        print("PRED :", pred["model_output"][:800])


Per-type score breakdown:
type                      n   mean  b10   b5   b1   b0  extr%
GSM_AnsAug              209   5.39  106    6   36   61 100.0%
GSM_FOBAR               122   3.46   38    1   37   46 100.0%
GSM_Rephrased           197   9.36  182    4    3    8 100.0%
GSM_SV                   97   4.93   45    2   18   32 100.0%
MATH_AnsAug             173   4.42   72    4   25   72  99.4%
MATH_FOBAR               45   3.93   16    1   12   16 100.0%
MATH_Rephrased          116   7.86   91    0    2   23 100.0%
MATH_SV                  41   3.98   15    1    8   17 100.0%

275 zero-score samples. Showing first 2:
IDX: 7 | type: GSM_FOBAR | rel_error: 1.7777777777777777
QUERY: Grace bắt đầu công việc kinh doanh cảnh quan của riêng mình. Cô tính phí 6 đô la một giờ cho việc cắt cỏ, 11 đô la cho việc nhổ cỏ và x đô la cho việc phủ lớp phủ. Vào tháng 9, cô cắt cỏ trong 63 giờ, nhổ cỏ trong 9 giờ và phủ lớp phủ trong 10 giờ. Cô ấy đã kiếm được bao nhiêu tiền vào tháng 9? Nếu chúng ta b

In [13]:
# ============================================================
# 11. Phase 2 — generate test_predictions.json
# ============================================================
# Set RUN_MODE = "phase2" at the top to activate this block.
if RUN_MODE == "phase2":
    if not TEST_FILE.exists():
        raise FileNotFoundError(f"Cannot find test.json at {TEST_FILE}")
    test_records = load_records(TEST_FILE)
    print("test:", len(test_records))

    # Always save pure model-only test output.
    model_test_outputs = generate_outputs(
        final_dir,
        test_records,
        MODEL_TEST_OUTPUT_PATH,
        max_new_tokens=MAX_NEW_TOKENS,
        mode=DECODE_MODE,
    )

    # Conservative retrieval can be applied as a post-processing step.
    # If you want pure model-only submission, set APPLY_RETRIEVAL_TO_TEST=False.
    if USE_SAFE_HYBRID_RETRIEVAL and APPLY_RETRIEVAL_TO_TEST:
        test_outputs, _ = apply_safe_hybrid_retrieval(
            test_records,
            model_test_outputs,
            TEST_OUTPUT_PATH,
            RETRIEVAL_INDEX_TRAIN,
            decision_report_path=TEST_RETRIEVAL_DECISIONS_PATH,
        )
    else:
        test_outputs = model_test_outputs
        write_outputs(TEST_OUTPUT_PATH, test_outputs)

    required_keys = {"id", "query_vi", "type", "model_output"}
    missing = [k for k in required_keys if k not in test_outputs[0]]
    if missing:
        raise ValueError(f"Output missing keys: {missing}")

    print("First test prediction:")
    print(json.dumps(test_outputs[0], ensure_ascii=False, indent=2)[:1500])


In [14]:
# ============================================================
# 12. List saved artifacts
# ============================================================
candidates = [
    VALID_OUTPUT_PATH,
    VALID_REPORT_PATH,
    MODEL_VALID_OUTPUT_PATH,
    MODEL_VALID_REPORT_PATH,
    HYBRID_VALID_OUTPUT_PATH,
    HYBRID_VALID_REPORT_PATH,
    VALID_RETRIEVAL_DECISIONS_PATH,
    Path(str(VALID_OUTPUT_PATH) + ".sha256.txt"),
    FINAL_OUTPUT_DIR / "model_hash.txt",
    TEST_OUTPUT_PATH,
    MODEL_TEST_OUTPUT_PATH,
    TEST_RETRIEVAL_DECISIONS_PATH,
    Path(str(TEST_OUTPUT_PATH) + ".sha256.txt"),
]
for p in candidates:
    p = Path(p)
    print(p, "| exists =", p.exists(), "| size =", p.stat().st_size if p.exists() else None)

print("\nWorking dir contents:")
for p in sorted(Path(WORKING_DIR).glob("*")):
    print("-", p)


/kaggle/working/valid_output.json | exists = True | size = 573864
/kaggle/working/valid_report.json | exists = True | size = 233795
/kaggle/working/model_valid_output.json | exists = True | size = 423182
/kaggle/working/model_valid_report.json | exists = True | size = 233949
/kaggle/working/hybrid_valid_output.json | exists = True | size = 573864
/kaggle/working/hybrid_valid_report.json | exists = True | size = 233795
/kaggle/working/valid_retrieval_decisions.json | exists = True | size = 23999
/kaggle/working/valid_output.json.sha256.txt | exists = True | size = 65
/kaggle/working/gpt2_math_bql_model_only_12ep_safe_retrieval/model_hash.txt | exists = True | size = 65
/kaggle/working/test_predictions.json | exists = False | size = None
/kaggle/working/model_test_predictions.json | exists = False | size = None
/kaggle/working/test_retrieval_decisions.json | exists = False | size = None
/kaggle/working/test_predictions.json.sha256.txt | exists = False | size = None

Working dir contents: